# Explainability Module Test

Test the `BaselineStatistics` and `AnomalyExplainer` classes on HDFS data.

**Goals:**
1. Fit baseline statistics on normal training data
2. Test explanation generation on known anomalies
3. Verify all analyzers work:
   - **Critical**: Explicit error keywords (Exception, Failed, Error)
   - **Frequency**: Event count deviations
   - **Pattern**: Unseen bigram transitions
   - **Position**: Events appearing in wrong sequence position
   - **Structure**: Abnormal sequence length
   - **Temporal**: Time gap anomalies (if timestamps available)
   - **Contextual**: Model surprise (deep learning)

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np

# Import our modules
from src.explainability import BaselineStatistics, AnomalyExplainer, format_anomaly_report
from src.utils.data_loader import load_loghub, create_train_val_test_split, filter_normal_samples

## 1. Load HDFS Data

In [2]:
DATA_DIR = '../data/hdfs/preprocessed'

# Load data
X, y, vocab = load_loghub(DATA_DIR)

print(f"Total sequences: {len(X)}")
print(f"Vocab size: {len(vocab)}")
print(f"Normal: {sum(y == 0)}, Anomaly: {sum(y == 1)}")

INFO:src.utils.data_loader:Loading data from LogHub preprocessing: ../data/hdfs/preprocessed
INFO:src.utils.data_loader:Loaded data:
INFO:src.utils.data_loader:  - Sequences: (575061,)
INFO:src.utils.data_loader:  - Labels: (575061,)
INFO:src.utils.data_loader:  - Normal: 558223, Anomaly: 16838
INFO:src.utils.data_loader:  - Loaded vocabulary: 29 events


Total sequences: 575061
Vocab size: 29
Normal: 558223, Anomaly: 16838


In [3]:
# Split data
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']

# Filter to normal samples for baseline fitting
X_train_normal, _ = filter_normal_samples(X_train, y_train, verbose=True)

INFO:src.utils.data_loader:Splitting data: train=0.7, val=0.15, test=0.15
INFO:src.utils.data_loader:Split complete:
INFO:src.utils.data_loader:  - Train: 402542 samples (11786 anomalies)
INFO:src.utils.data_loader:  - Val:   86259 samples (2526 anomalies)
INFO:src.utils.data_loader:  - Test:  86260 samples (2526 anomalies)
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:FILTERING TRAINING DATA FOR SEMI-SUPERVISED LEARNING
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:Original training size: 402542 samples
INFO:src.utils.data_loader:  Normal samples: 390,756 (97.07%)
INFO:src.utils.data_loader:  Anomaly samples: 11,786 (2.93%)
INFO:src.utils.data_loader:
Filtered training size: 390,756 samples (NORMAL ONLY)
INFO:src.utils.data_loader:Removed 11,786 anomalies from training set
INFO:src.utils.data_loader:✓ Training data is now pure no

## 2. Fit Baseline Statistics

In [4]:
# Create and fit baseline
baseline = BaselineStatistics(ngram_orders=[2, 3])
baseline.fit(X_train_normal, vocab)

INFO:src.explainability.baseline_stats:Fitting baseline statistics on 390756 sequences...
INFO:src.explainability.baseline_stats:Computing template frequency statistics...
INFO:src.explainability.baseline_stats:Computing n-gram statistics for orders [2, 3]...
INFO:src.explainability.baseline_stats:Computing positional statistics...
INFO:src.explainability.baseline_stats:Computed positional stats for 17 templates.
INFO:src.explainability.baseline_stats:Baseline statistics fitted successfully.



[Baseline Stats] Analyzed 390756 sequences.
[Baseline Stats] Avg Length: 19.5 (±4.8)
[Baseline Stats] Position Patterns: 17 templates.
[Baseline Stats] N-Gram Patterns: 645.
[Baseline Stats] Time Patterns: 0 transitions.


In [5]:
# Inspect template statistics
print("Template Statistics (sample):")
print("-" * 50)
for i, (template, stats) in enumerate(baseline.template_stats.items()):
    if i >= 5: break
    print(f"{template}: mean={stats['mean']:.2f}, std={stats['std']:.2f}, max={stats['max']}, presence={stats['presence_rate']:.2%}")

Template Statistics (sample):
--------------------------------------------------
E1: mean=0.00, std=0.00, max=0, presence=0.00%
E10: mean=0.00, std=0.00, max=0, presence=0.00%
E11: mean=3.00, std=0.00, max=3, presence=100.00%
E12: mean=0.00, std=0.00, max=0, presence=0.00%
E13: mean=0.00, std=0.00, max=0, presence=0.00%


In [6]:
# Inspect n-gram statistics
print("\nBigram Statistics (top 10 by frequency):")
print("-" * 50)

if 2 in baseline.ngram_counts:
    sorted_bigrams = sorted(baseline.ngram_counts[2].items(), key=lambda x: x[1], reverse=True)[:10]
    for bigram, count in sorted_bigrams:
        prob = baseline.ngram_stats[2].get(bigram, 0)
        print(f"{bigram[0]} -> {bigram[1]}: count={count}, prob={prob:.3f}")


Bigram Statistics (top 10 by frequency):
--------------------------------------------------
E11 -> E9: count=1156695, prob=0.987
E26 -> E26: count=700140, prob=0.631
E9 -> E11: count=693886, prob=0.592
E23 -> E23: count=636345, prob=0.667
E21 -> E21: count=635961, prob=0.998
E5 -> E5: count=621346, prob=0.529
E9 -> E26: count=396389, prob=0.338
E23 -> E21: count=317557, prob=0.333
E5 -> E22: count=291776, prob=0.249
E22 -> E5: count=257358, prob=0.659


In [7]:
# Sequence-level stats
print("\nSequence Statistics:")
print("-" * 50)
for key, val in baseline.sequence_stats.items():
    print(f"{key}: {val}")


Sequence Statistics:
--------------------------------------------------
length_mean: 19.501453592523212
length_std: 4.782233780498785
length_min: 13
length_max: 298
num_sequences: 390756


## 3. Test AnomalyExplainer

In [8]:
# Load event templates for Critical keyword detection
import pandas as pd

templates_df = pd.read_csv(f'{DATA_DIR}/HDFS.log_templates.csv')
event_templates = dict(zip(templates_df['EventId'], templates_df['EventTemplate']))

print(f"Loaded {len(event_templates)} event templates")

# Create explainer WITH event templates (enables Critical keyword detection)
explainer = AnomalyExplainer(baseline, event_templates=event_templates)
print("✓ AnomalyExplainer created with Critical keyword detection enabled")

Loaded 29 event templates
✓ AnomalyExplainer created with Critical keyword detection enabled


In [9]:
# Find some actual anomalies from test set
anomaly_indices = np.where(y_test == 1)[0]
normal_indices = np.where(y_test == 0)[0]

print(f"Anomalies in test set: {len(anomaly_indices)}")
print(f"Normal in test set: {len(normal_indices)}")

Anomalies in test set: 2526
Normal in test set: 83734


In [10]:
# Test on a few anomalies
print("=" * 60)
print("EXPLAINING ANOMALOUS SEQUENCES")
print("=" * 60)

for i, idx in enumerate(anomaly_indices[:5]):
    seq = X_test[idx]
    
    print(f"\n--- Anomaly #{i+1} (index {idx}) ---")
    print(f"Sequence: {list(seq)[:10]}..." if len(seq) > 10 else f"Sequence: {list(seq)}")
    print(f"Length: {len(seq)}")
    
    # Get explanation
    explanation = explainer.explain(seq, top_k=3)
    
    if explanation['is_explained']:
        print("Reasons:")
        for reason in explanation['reasons']:
            print(f"  - {reason}")
    else:
        print("  No statistical anomalies detected (model-only detection)")

EXPLAINING ANOMALOUS SEQUENCES

--- Anomaly #1 (index 24) ---
Sequence: ['E5', 'E22']
Length: 2
Reasons:
  - Event 'E22' at end (pos 1.00), normally at start (mean 0.10)
  - Sequence is abnormally short (length 2, normal range: 10-29)

--- Anomaly #2 (index 156) ---
Sequence: ['E22', 'E5', 'E5', 'E7']
Length: 4
Reasons:
  - Unexpected pattern: [E5 -> E7] never seen in training
  - Event 'E5' at end (pos 0.67), normally at start (mean 0.08)
  - Event 'E5' at middle (pos 0.33), normally at start (mean 0.08)

--- Anomaly #3 (index 157) ---
Sequence: ['E5', 'E5', 'E5', 'E22', 'E9', 'E11', 'E9', 'E11', 'E9', 'E26']...
Length: 20
  No statistical anomalies detected (model-only detection)

--- Anomaly #4 (index 160) ---
Sequence: ['E5', 'E5', 'E5', 'E22', 'E11', 'E9', 'E11', 'E9', 'E11', 'E9']...
Length: 20
  No statistical anomalies detected (model-only detection)

--- Anomaly #5 (index 223) ---
Sequence: ['E5', 'E5', 'E22', 'E7']
Length: 4
Reasons:
  - Unexpected pattern: [E22 -> E7] never 

In [11]:
# Test on normal sequences (should have few/no explanations)
print("=" * 60)
print("CHECKING NORMAL SEQUENCES (Should have minimal reasons)")
print("=" * 60)

for i, idx in enumerate(normal_indices[:5]):
    seq = X_test[idx]
    explanation = explainer.explain(seq, top_k=3)
    
    print(f"\n--- Normal #{i+1} ---")
    print(f"Explained: {explanation['is_explained']}, Reasons: {len(explanation['details'])}")

CHECKING NORMAL SEQUENCES (Should have minimal reasons)

--- Normal #1 ---
Explained: False, Reasons: 0

--- Normal #2 ---
Explained: False, Reasons: 0

--- Normal #3 ---
Explained: False, Reasons: 0

--- Normal #4 ---
Explained: False, Reasons: 0

--- Normal #5 ---
Explained: False, Reasons: 0


## 4. Analyze Explanation Coverage

In [12]:
# Check how many anomalies can be explained (Stats-only, no model)
from tqdm import tqdm

explained_count = 0
reason_types = {'Critical': 0, 'Frequency': 0, 'Pattern': 0, 'Temporal': 0, 'Position': 0, 'Structure': 0}

for idx in tqdm(anomaly_indices, desc="Analyzing anomalies"):
    seq = X_test[idx]
    explanation = explainer.explain(seq, top_k=10)
    
    if explanation['is_explained']:
        explained_count += 1
        for detail in explanation['details']:
            if detail['type'] in reason_types:
                reason_types[detail['type']] += 1

print(f"\n" + "=" * 50)
print(f"STATS-ONLY EXPLANATION COVERAGE")
print(f"=" * 50)
print(f"Total anomalies: {len(anomaly_indices)}")
print(f"Explained: {explained_count} ({explained_count/len(anomaly_indices)*100:.1f}%)")
print(f"\nReason breakdown:")
for rtype, count in reason_types.items():
    print(f"  {rtype}: {count}")

Analyzing anomalies: 100%|██████████| 2526/2526 [00:00<00:00, 142889.29it/s]


STATS-ONLY EXPLANATION COVERAGE
Total anomalies: 2526
Explained: 1896 (75.1%)

Reason breakdown:
  Critical: 0
  Frequency: 2116
  Pattern: 1187
  Temporal: 0
  Position: 2326
  Structure: 1010


## 5. Save Baseline for Later Use

In [13]:
# Save baseline statistics
save_path = '../data/hdfs/preprocessed/baseline_stats.pkl'
baseline.save(save_path)
print(f"Baseline saved to {save_path}")

INFO:src.explainability.baseline_stats:Baseline saved to ../data/hdfs/preprocessed/baseline_stats.pkl


Baseline saved to ../data/hdfs/preprocessed/baseline_stats.pkl


In [14]:
# Test loading
baseline_loaded = BaselineStatistics.load(save_path)
print(f"Loaded baseline with {len(baseline_loaded.template_stats)} templates")

Loaded baseline with 29 templates


## 6. Test Model Surprise (Contextual Analysis)

Load the trained LogGPT model and test the full hybrid explanation pipeline.

In [15]:
# Load the trained LogGPT model
import torch
from src.models.loggptmodel import LogGPTModel
from src.engine.trainer import LogSeqTrainer

# Device setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

# Load semantic embeddings (same as training)
semantic_vectors = torch.load(f'{DATA_DIR}/semantic_embeddings.pt')
emb_dim = semantic_vectors.shape[1]

# Recreate model architecture (must match training)
model = LogGPTModel(
    vocab_size=len(vocab),
    embedding_dim=emb_dim,
    num_layers=6,
    num_heads=8,
    dropout=0.1,
    semantic_embeddings=semantic_vectors
)

# Load trained weights
checkpoint_path = '../mdls/loggpt_checkpoint.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Loaded LogGPT model from {checkpoint_path}")

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Using device: mps
LogGPT: Loaded semantic embeddings directly (384d).
✓ Loaded LogGPT model from ../mdls/loggpt_checkpoint.pt


In [16]:
# Test explain_full on a few anomalies
print("=" * 70)
print("FULL HYBRID EXPLANATION (Statistics + Model Surprise)")
print("=" * 70)

# Need to convert sequences to integer IDs
def to_ids(seq, vocab):
    return [vocab[e] for e in seq]

for i, idx in enumerate(anomaly_indices[:5]):
    seq = X_test[idx]
    seq_ids = to_ids(seq, vocab)
    
    print(f"\n--- Anomaly #{i+1} (index {idx}) ---")
    print(f"Sequence: {list(seq)[:8]}..." if len(seq) > 8 else f"Sequence: {list(seq)}")
    
    # Use explain_full with model
    explanation = explainer.explain_full(
        sequence=seq_ids,
        model=model,
        device=device,
        top_k=5,
        surprise_threshold=0.10
    )
    
    print(f"Sources: {explanation['explanation_sources']}")
    print(f"Stats reasons: {explanation['stat_reasons_count']}, Model reasons: {explanation['model_reasons_count']}")
    
    if explanation['is_explained']:
        print("Top Reasons:")
        for reason in explanation['reasons'][:3]:
            print(f"  - {reason}")

FULL HYBRID EXPLANATION (Statistics + Model Surprise)

--- Anomaly #1 (index 24) ---
Sequence: ['E5', 'E22']
Sources: ['Structure', 'Position']
Stats reasons: 2, Model reasons: 0
Top Reasons:
  - Sequence is abnormally short (length 2, normal range: 10-29)
  - Event 'E22' at end (pos 1.00), normally at start (mean 0.10)

--- Anomaly #2 (index 156) ---
Sequence: ['E22', 'E5', 'E5', 'E7']
Sources: ['Position', 'Pattern', 'Contextual', 'Critical', 'Structure']
Stats reasons: 4, Model reasons: 1
Top Reasons:
  - Explicit error at step 4: 'E7' contains 'EXCEPTION'
  - Sequence is abnormally short (length 4, normal range: 10-29)
  - Event 'E5' at end (pos 0.67), normally at start (mean 0.08)

--- Anomaly #3 (index 157) ---
Sequence: ['E5', 'E5', 'E5', 'E22', 'E9', 'E11', 'E9', 'E11']...
Sources: ['Contextual', 'Critical']
Stats reasons: 0, Model reasons: 2
Top Reasons:
  - Explicit error at step 19: 'E20' contains 'ERROR'
  - Step 18: Model expected [E21(1.00), E2(0.00), E4(0.00)] but got 'E

In [17]:
# Full coverage analysis with Model Surprise + Critical Keywords
from tqdm import tqdm

explained_count = 0
reason_types = {'Critical': 0, 'Frequency': 0, 'Pattern': 0, 'Temporal': 0, 'Position': 0, 'Structure': 0, 'Contextual': 0}

# Track explanation sources
stats_only_explained = 0
model_assisted_explained = 0
critical_detected = 0

for idx in tqdm(anomaly_indices, desc="Analyzing with model"):
    seq = X_test[idx]
    seq_ids = to_ids(seq, vocab)
    
    explanation = explainer.explain_full(
        sequence=seq_ids,
        model=model,
        device=device,
        top_k=10,
        surprise_threshold=0.10
    )
    
    if explanation['is_explained']:
        explained_count += 1
        
        # Track critical detections
        if explanation.get('critical_reasons_count', 0) > 0:
            critical_detected += 1
        
        # Track if model was needed
        if explanation['model_reasons_count'] > 0 and explanation['stat_reasons_count'] == 0:
            model_assisted_explained += 1
        elif explanation['stat_reasons_count'] > 0:
            stats_only_explained += 1
        
        for detail in explanation['details']:
            if detail['type'] in reason_types:
                reason_types[detail['type']] += 1

print(f"\n" + "=" * 60)
print(f"FULL HYBRID EXPLANATION COVERAGE REPORT")
print(f"=" * 60)
print(f"Total anomalies: {len(anomaly_indices)}")
print(f"Explained: {explained_count} ({explained_count/len(anomaly_indices)*100:.1f}%)")
print(f"\nBreakdown:")
print(f"  - Stats-only explained: {stats_only_explained}")
print(f"  - Model-assisted (stats failed): {model_assisted_explained}")
print(f"  - Critical keyword detected: {critical_detected}")
print(f"\nReason types:")
for rtype, count in reason_types.items():
    print(f"  {rtype}: {count}")

Analyzing with model: 100%|██████████| 2526/2526 [00:32<00:00, 77.88it/s] 


FULL HYBRID EXPLANATION COVERAGE REPORT
Total anomalies: 2526
Explained: 2526 (100.0%)

Breakdown:
  - Stats-only explained: 1896
  - Model-assisted (stats failed): 630
  - Critical keyword detected: 1615

Reason types:
  Critical: 2635
  Frequency: 1630
  Pattern: 1143
  Temporal: 0
  Position: 2438
  Structure: 1072
  Contextual: 3899


In [18]:
# Demo: format_anomaly_report with Critical keywords
print("=" * 70)
print("FORMATTED ANOMALY REPORTS (with Critical keyword override)")
print("=" * 70)

# Find anomalies with exception events (E4, E7, E10, E12, E14)
exception_events = {'E4', 'E7', 'E10', 'E12', 'E14'}

for i, idx in enumerate(anomaly_indices[:20]):
    seq = X_test[idx]
    
    # Check if this sequence has any exception events
    if not any(e in exception_events for e in seq):
        continue
    
    seq_ids = to_ids(seq, vocab)
    
    explanation = explainer.explain_full(
        sequence=seq_ids,
        model=model,
        device=device,
        top_k=5,
        surprise_threshold=0.10
    )
    
    # Use the formatter
    report = format_anomaly_report(
        sequence_id=f"test_idx_{idx}",
        anomaly_score=0.99,  # placeholder
        explanation_result=explanation,
        max_reasons=5
    )
    print(report)
    print()
    
    # Only show first 3 examples
    if i >= 2:
        break

FORMATTED ANOMALY REPORTS (with Critical keyword override)
ANOMALY DETECTED: test_idx_156
Confidence: 99.00%

BECAUSE:
  [CRITICAL] Explicit error at step 4: 'E7' contains 'EXCEPTION'
  [STRUCTURE] Sequence is abnormally short (length 4, normal range: 10-29)
  [POSITION] Event 'E5' at end (pos 0.67), normally at start (mean 0.08)
  [POSITION] Event 'E5' at middle (pos 0.33), normally at start (mean 0.08)
  [PATTERN] Unexpected pattern: [E5 -> E7] never seen in training

Sources: Position, Pattern, Contextual, Critical, Structure

ANOMALY DETECTED: test_idx_223
Confidence: 99.00%

BECAUSE:
  [CRITICAL] Explicit error at step 4: 'E7' contains 'EXCEPTION'
  [STRUCTURE] Sequence is abnormally short (length 4, normal range: 10-29)
  [POSITION] Event 'E22' at end (pos 0.67), normally at start (mean 0.10)
  [POSITION] Event 'E5' at middle (pos 0.33), normally at start (mean 0.08)
  [PATTERN] Unexpected pattern: [E22 -> E7] never seen in training

Sources: Position, Pattern, Contextual, Critic

## 7. Trust Test: Qualitative Validation

Manually verify that generated explanations match what a human would identify as the anomaly cause.

In [19]:
# Display event templates (already loaded in cell-11)
print("Event ID -> Template Mapping:")
print("-" * 80)
for eid, template in list(event_templates.items())[:10]:
    print(f"{eid}: {template[:70]}...")

Event ID -> Template Mapping:
--------------------------------------------------------------------------------
E1: [*]Adding an already existing block[*]...
E2: [*]Verification succeeded for[*]...
E3: [*]Served block[*]to[*]...
E4: [*]Got exception while serving[*]to[*]...
E5: [*]Receiving block[*]src:[*]dest:[*]...
E6: [*]Received block[*]src:[*]dest:[*]of size[*]...
E7: [*]writeBlock[*]received exception[*]...
E8: [*]PacketResponder[*]for block[*]Interrupted[*]...
E9: [*]Received block[*]of size[*]from[*]...
E10: [*]PacketResponder[*]Exception[*]...


In [20]:
def display_trust_test_case(idx, seq, explanation, event_templates, case_num):
    """Display a single anomaly for manual validation."""
    
    print(f"\n{'='*80}")
    print(f"TRUST TEST CASE #{case_num} (Test Index: {idx})")
    print(f"{'='*80}")
    
    # 1. Show sequence with human-readable templates
    print(f"\n[SEQUENCE] Length: {len(seq)}")
    print("-" * 40)
    for i, event in enumerate(seq):
        template = event_templates.get(event, "Unknown")
        # Truncate long templates
        template_short = template[:60] + "..." if len(template) > 60 else template
        print(f"  {i+1:2d}. {event}: {template_short}")
    
    # 2. Show generated explanations
    print(f"\n[MACHINE EXPLANATIONS] Sources: {explanation.get('explanation_sources', [])}")
    print("-" * 40)
    if explanation['is_explained']:
        for i, detail in enumerate(explanation['details'][:5]):
            print(f"  {i+1}. [{detail['type']}] {detail['message']}")
    else:
        print("  No explanations generated")
    
    # 3. Prompt for human assessment
    print(f"\n[YOUR ASSESSMENT]")
    print("-" * 40)
    print("  Q1: What do YOU think is wrong with this sequence?")
    print("  Q2: Does the machine explanation match your intuition? (Yes/No/Partial)")
    print()

# Sample random anomalies for validation
np.random.seed(42)
sample_size = 20
sample_indices = np.random.choice(anomaly_indices, size=sample_size, replace=False)

print(f"Sampled {sample_size} anomalies for Trust Test")
print(f"Indices: {list(sample_indices[:5])}...")

Sampled 20 anomalies for Trust Test
Indices: [np.int64(67159), np.int64(26118), np.int64(7234), np.int64(33498), np.int64(45156)]...


In [26]:
# Run Trust Test on first 5 samples (run more as needed)
# Change the range to see more cases: range(5, 10), range(10, 15), etc.

for case_num, idx in enumerate(sample_indices[5:10], 1):
    seq = X_test[idx]
    seq_ids = to_ids(seq, vocab)
    
    # Get full explanation
    explanation = explainer.explain_full(
        sequence=seq_ids,
        model=model,
        device=device,
        top_k=5,
        surprise_threshold=0.10
    )
    
    display_trust_test_case(idx, seq, explanation, event_templates, case_num)


TRUST TEST CASE #1 (Test Index: 6302)

[SEQUENCE] Length: 28
----------------------------------------
   1. E5: [*]Receiving block[*]src:[*]dest:[*]
   2. E5: [*]Receiving block[*]src:[*]dest:[*]
   3. E22: [*]BLOCK* NameSystem[*]allocateBlock:[*]
   4. E5: [*]Receiving block[*]src:[*]dest:[*]
   5. E11: [*]PacketResponder[*]for block[*]terminating[*]
   6. E9: [*]Received block[*]of size[*]from[*]
   7. E11: [*]PacketResponder[*]for block[*]terminating[*]
   8. E9: [*]Received block[*]of size[*]from[*]
   9. E11: [*]PacketResponder[*]for block[*]terminating[*]
  10. E9: [*]Received block[*]of size[*]from[*]
  11. E26: [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]i...
  12. E26: [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]i...
  13. E25: [*]BLOCK* ask[*]to replicate[*]to[*]
  14. E5: [*]Receiving block[*]src:[*]dest:[*]
  15. E18: [*]Starting thread to transfer block[*]to[*]
  16. E16: [*]:Transmitted block[*]to[*]
  17. E6: [*]Received block[*]src:[*]dest

In [24]:
# Record your validation results here
# After reviewing each case, mark: "Yes" (match), "No" (mismatch), "Partial" (somewhat)

validation_results = {
    # Case #: (Match?, Your Notes)
    
    # Case 1: "Unexpected error trying to delete block" (E20)
    # Result: Caught by Critical Keyword check.
    1: ("Yes", "Correctly flagged explicit 'Unexpected error' (E20) via Critical Tier check."),
    
    # Case 2: Logic Violation (Adding a block while Deleting it)
    # Result: Caught by Contextual Model (Low Probability/Surprise).
    2: ("Yes", "Contextual model correctly identified invalid state transition (Add after Delete)."),
    
    # Case 3: "Unexpected error" (E20) - Simple case
    # Result: Caught by Critical Keyword check.
    3: ("Yes", "Correctly flagged explicit error keyword in E20."),
    
    # Case 4: Lifecycle Ordering (Receiving before Allocating)
    # Result: Caught by Positional Stats.
    4: ("Yes", "Positional analyzer correctly flagged E5 appearing too early/out of order."),
    
    # Case 5: Exception (E4) hidden by massive replication noise
    # Result: Caught by Critical Priority Override (The fix we just made).
    5: ("Yes", "Critical priority override successfully lifted the buried 'Exception' (E4) to the top."),
    
    # Add more as you review cases 6-20
}

# Calculate match rate
matches = sum(1 for v in validation_results.values() if v[0] == "Yes")
partials = sum(1 for v in validation_results.values() if v[0] == "Partial")
total = len(validation_results)

print("==========================================")
print(f"VALIDATION SUMMARY (N={total})")
print("==========================================")
print(f"Exact Matches:   {matches}/{total} ({matches/total*100:.1f}%)")
print(f"Partial Matches: {partials}/{total}")
print(f"Failures:        {total - matches - partials}/{total}")

VALIDATION SUMMARY (N=5)
Exact Matches:   5/5 (100.0%)
Partial Matches: 0/5
Failures:        0/5


In [23]:
# HDFS Block Lifecycle Reference (for manual validation)
print("""
================================================================================
HDFS BLOCK LIFECYCLE REFERENCE
================================================================================

NORMAL BLOCK FLOW:
  1. E22 (allocateBlock)     → Block is created/allocated
  2. E5  (Receiving block)   → Data is being received
  3. E9  (Received block)    → Data reception complete
  4. E26 (addStoredBlock)    → Block added to storage
  5. E11 (PacketResponder terminating) → Write operation ends
  6. E21 (Deleting block)    → Block deletion (cleanup)

COMMON ANOMALY PATTERNS:
  - Missing E22 at start     → Block allocation missing
  - E7 (writeBlock exception)→ Write failure
  - E4 (exception serving)   → Read failure  
  - E14 (exception receive)  → Receive failure
  - Very short sequence      → Premature termination
  - E5 without E9            → Incomplete data transfer

KEY EVENTS:
  - E5/E6/E9: Data transfer events
  - E7/E10/E12/E14: Exception/error events
  - E11: Normal termination
  - E21/E23: Deletion events
  - E22/E26: Block management events
================================================================================
""")


HDFS BLOCK LIFECYCLE REFERENCE

NORMAL BLOCK FLOW:
  1. E22 (allocateBlock)     → Block is created/allocated
  2. E5  (Receiving block)   → Data is being received
  3. E9  (Received block)    → Data reception complete
  4. E26 (addStoredBlock)    → Block added to storage
  5. E11 (PacketResponder terminating) → Write operation ends
  6. E21 (Deleting block)    → Block deletion (cleanup)

COMMON ANOMALY PATTERNS:
  - Missing E22 at start     → Block allocation missing
  - E7 (writeBlock exception)→ Write failure
  - E4 (exception serving)   → Read failure  
  - E14 (exception receive)  → Receive failure
  - Very short sequence      → Premature termination
  - E5 without E9            → Incomplete data transfer

KEY EVENTS:
  - E5/E6/E9: Data transfer events
  - E7/E10/E12/E14: Exception/error events
  - E11: Normal termination
  - E21/E23: Deletion events
  - E22/E26: Block management events

